### 3.2.3. Smooth Optimization

$$
f \in C^2,\qquad
f(\mathbf{x}+\mathbf{p}) = f(\mathbf{x}) + \nabla f(\mathbf{x})^\top \mathbf{p} + \tfrac12 \mathbf{p}^\top \nabla^2 f(\mathbf{x})\, \mathbf{p} + o(\|\mathbf{p}\|^2).
$$

$$
\nabla f(\mathbf{x}^\star) = \mathbf 0,\qquad \nabla^2 f(\mathbf{x}^\star) \succeq 0.
$$

**Explanation:**

Smooth optimization treats objectives that are continuously differentiable, so a gradient $\nabla f$ and (for second-order methods) a Hessian $\nabla^2 f$ exist and vary continuously everywhere. Smoothness is what makes the second-order Taylor expansion a faithful *local quadratic model* of the objective, and that model is the engine behind gradient, [Newton](../09_Algorithms/11_newtons_method.ipynb), and quasi-Newton methods. The defining contrast is with [nonsmooth optimization](./04_nonsmooth_optimization.ipynb), where kinks force the gradient to be replaced by a subdifferential.

**Properties:**
- The first-order condition $\nabla f(\mathbf{x}^\star)=\mathbf 0$ locates stationary points.
- A positive semidefinite Hessian at a stationary point certifies a local minimum.
- Near a minimizer the objective looks like a convex quadratic, which is why smooth methods converge quickly there.

**Numerical Example:**

Take the smooth, non-quadratic objective
$$
f(\mathbf{x}) = e^{x_1} - x_1 + x_2^2.
$$

Its gradient and Hessian are
$$
\nabla f(\mathbf{x}) = \begin{bmatrix} e^{x_1} - 1 \\ 2x_2 \end{bmatrix},
\qquad
\nabla^2 f(\mathbf{x}) = \begin{bmatrix} e^{x_1} & 0 \\ 0 & 2 \end{bmatrix}.
$$

Setting the gradient to zero gives $e^{x_1}=1$ and $x_2=0$, so $\mathbf{x}^\star=(0,0)$. The Hessian there is $\operatorname{diag}(1,2)\succ 0$, so the point is a local minimum, and the second-order model around it is
$$
f(\mathbf{p}) \approx 1 + \tfrac12\big(p_1^2 + 2p_2^2\big).
$$
Both derivatives exist and are continuous everywhere, which is exactly what "smooth" provides.

In [1]:
import sympy as sp

x_1, x_2 = sp.symbols("x_1 x_2", real=True)
variables = [x_1, x_2]
objective = sp.exp(x_1) - x_1 + x_2**2

gradient = sp.Matrix([sp.diff(objective, variable) for variable in variables])
hessian = sp.hessian(objective, variables)
stationary_solution = sp.solve(gradient, variables, dict=True)[0]
minimizer = tuple(stationary_solution[variable] for variable in variables)
hessian_at_minimizer = hessian.subs(stationary_solution)

print("gradient ="); sp.pprint(gradient)
print("hessian ="); sp.pprint(hessian)
print("minimizer =", minimizer)
print("Hessian eigenvalues at minimizer =", list(hessian_at_minimizer.eigenvals()))

gradient =
⎡ x₁    ⎤
⎢ℯ   - 1⎥
⎢       ⎥
⎣ 2⋅x₂  ⎦
hessian =
⎡ x₁   ⎤
⎢ℯ    0⎥
⎢      ⎥
⎣ 0   2⎦
minimizer = (0, 0)
Hessian eigenvalues at minimizer = [1, 2]


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
objective = ca.exp(decision_variable[0]) - decision_variable[0] + decision_variable[1]**2
hessian, gradient = ca.hessian(objective, decision_variable)

solver = ca.nlpsol("solver", "ipopt", {"x": decision_variable, "f": objective},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})
solution = solver(x0=[1, 1])

print("minimizer =", solution["x"])
print("gradient at minimizer =", ca.evalf(ca.substitute(gradient, decision_variable, solution["x"])))
print("Hessian at minimizer =", ca.evalf(ca.substitute(hessian, decision_variable, solution["x"])))

minimizer = [1.22332e-12, 0]
gradient at minimizer = [1.22324e-12, 0]
Hessian at minimizer = 
[[1, 00], 
 [00, 2]]


**References:**

[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)  
[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.). Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Constrained Optimization](./02_constrained_optimization.ipynb) | [Next: Nonsmooth Optimization ➡️](./04_nonsmooth_optimization.ipynb)